# Baseline: PCA Reconstruction Error

Before training any autoencoder, we need a reference point: how well does a simple, non-deep method do on the same data?

**Why PCA?** PCA is the linear version of an autoencoder. It compresses each row to a few principal components and reconstructs it back. Like an autoencoder trained on benign traffic only, it reconstructs normal traffic well and attacks poorly, so the reconstruction error is the anomaly score.

If an autoencoder cannot beat PCA, the non-linear (deep) part adds nothing. Uses the shared data and `evaluate.py`, exactly like the autoencoder notebooks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

DATA_DIR = "/content/drive/MyDrive/IoMT_project/data"
sys.path.insert(0, DATA_DIR)
from evaluate import load_data, evaluate, compare_percentiles

SEED = 42
X_train, X_val, X_test, y_test, attack_test = load_data(DATA_DIR)

## 1. Choosing the number of components

We keep the smallest number of components that explains 95% of the variance of the benign training data. This plays the same role as the bottleneck size in an autoencoder.

In [ ]:
full = PCA(random_state=SEED).fit(X_train)
cum = np.cumsum(full.explained_variance_ratio_)
k = int(np.searchsorted(cum, 0.95) + 1)

plt.figure(figsize=(7, 3.5))
plt.plot(range(1, len(cum) + 1), cum, marker="o", ms=3)
plt.axhline(0.95, color="k", ls="--", lw=0.8)
plt.axvline(k, color="k", ls=":", lw=0.8)
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.title(f"{k} components explain 95% of the variance")
plt.tight_layout(); plt.savefig(f"{DATA_DIR}/figures/pca_explained_variance.png", dpi=120); plt.show()

## 2. Fit on benign training data and score

In [ ]:
pca = PCA(n_components=k, random_state=SEED).fit(X_train)

def pca_errors(X):
    """Per-row MSE between the input and its PCA reconstruction."""
    recon = pca.inverse_transform(pca.transform(X))
    return np.mean((X - recon) ** 2, axis=1)

val_scores = pca_errors(X_val)
test_scores = pca_errors(X_test)
print(f"components: {k} | input features: {X_train.shape[1]}")

## 3. Evaluation

Same rule as every model: the threshold is the 99th percentile of the validation (benign) scores.

In [ ]:
res = evaluate(test_scores, y_test, attack_test, val_scores,
               model_name="Baseline: PCA", person="baseline", save_dir=DATA_DIR)

In [ ]:
compare_percentiles(test_scores, y_test, val_scores);

## 4. What the autoencoders need to beat

The table below is the target. The attacks with the lowest recall are where a non-linear model can add value.

In [ ]:
per_attack = pd.Series(res["per_attack_recall"]).sort_values()
print(f"F1 to beat: {res['f1']:.3f} | PR-AUC to beat: {res['pr_auc']:.3f}\n")
print("Weakest attacks for PCA:")
print(per_attack.head(8).to_string(float_format="{:.3f}".format))